In [21]:
from datetime import datetime
import random
class Player:
    def __init__(self,_id,name,_hp):
        self._id=_id
        self._name=name.strip().title()
        self._hp=max(0,_hp)
        self._inventory=[]
    def __str__(self):
        return f"Player(id={self._id},name={self._name},hp={self._hp})"
    def __del__(self):
        print(f"Player {self._name} deleted")
    def from_str(cls,data):
        parts=[x.strip() for x in data.split(",")]
        if len(parts)!=3:
            raise ValueError("Error in string")
        return cls(int(parts[0]),parts[1],int(parts[2]))
    def handle_event(self,event):
        if event.type=="attack":
            self._hp-=event.data.get("damage",0)
        elif event.type=="heal":
            self._hp+=event.data.get("heal",0)
        elif event.type=="loot":
            item=event.data.get("item")
            if item:
                self._inventory.append(item)
class Warrior(Player):
    def handle_event(self,event):
        if event.type=="attack":
            dmg=event.data.get("damage",0)*0.9
            self._hp-=int(dmg)
        else:
            super().handle_event(event)
class Mage(Player):
    def handle_event(self,event):
        if event.type=="loot":
            item=event.data.get("item")
            if item:
                item.power=int(item.power*1.1)
        super().handle_event(event)

class Item:
    def __init__(self,_id,name,power):
        self.id=_id
        self.name=name.strip().title()
        self.power=power

    def __str__(self):
        return f"Item(id={self.id},name={self.name},power={self.power})"
class Event:
    def __init__(self,type_,data):
        self.type=type_
        self.data=data
        self.timestamp=datetime.now()
    def __str__(self):
        return f"{self.type}|{self.data}|{self.timestamp}"

def generate_events(players,items,n):
    events=[]
    types=["attack","heal","loot"]
    for p in players:
        for _ in range(n):
            t=random.choice(types)

            if t=="attack":
                events.append(Event(t,{"damage":random.randint(5,20),"player":p._id}))
            elif t=="heal":
                events.append(Event(t,{"heal":random.randint(5,15),"player":p._id}))
            else:
                events.append(Event(t,{"item":random.choice(items),"player":p._id}))
    return events
def damage_stream(events):
    for e in events:
        if e.type=="attack":
            yield e.data.get("damage",0)

def analyze_logs(events):
    total_damage=sum(e.data.get("damage",0) for e in events if e.type=="attack")
    return {"total_damage":total_damage}
decide_action=lambda player:(
    "heal" if player._hp<30 else
    "loot" if len(player._inventory)<2 else
    "attack"
)
def main():
    players=[
        Warrior(1,"john",100),
        Mage(2,"alice",80)
    ]
    items=[
        Item(1,"mech",50),
        Item(2,"palka",40)
    ]
    events=generate_events(players,items,3)

    print("\nEVENTS:")
    for e in events:
        print(e)
    for e in events:
        for p in players:
            if e.data.get("player")==p._id:
                p.handle_event(e)

    print("\nPLAYERS(AFTER EVENT):")
    for p in players:
        print(p)

    print("\nHEAT")
    for dmg in damage_stream(events):
        print(dmg)

    stats=analyze_logs(events)
    print("\nSTATISTICS")
    print(stats)

    print("\nAT DECITION")
    for p in players:
        print(p._name,"---->",decide_action(p))

if __name__=="__main__":
    main()


EVENTS:
attack|{'damage': 13, 'player': 1}|2026-04-11 04:51:18.604511
loot|{'item': <__main__.Item object at 0x11200fc50>, 'player': 1}|2026-04-11 04:51:18.604516
attack|{'damage': 9, 'player': 1}|2026-04-11 04:51:18.604518
heal|{'heal': 13, 'player': 2}|2026-04-11 04:51:18.604525
attack|{'damage': 10, 'player': 2}|2026-04-11 04:51:18.604527
heal|{'heal': 10, 'player': 2}|2026-04-11 04:51:18.604529

PLAYERS(AFTER EVENT):
Player(id=1,name=John,hp=81)
Player(id=2,name=Alice,hp=93)

HEAT
13
9
10

STATISTICS
{'total_damage': 32}

AT DECITION
John ----> loot
Alice ----> loot
Player John deleted
Player Alice deleted
